# DCC Pipeline Demo

This notebook is a lightweight instructor-facing demo for the Duke Compute Cluster workflow.

It is designed to run **after** the main pipeline stages have produced artifacts:

1. `render-batch`
2. `convert`
3. `sanity-check`
4. `train`
5. `evaluate`

The notebook does not retrigger expensive jobs. It reads existing DCC outputs and shows:

- one input kitchen image
- one rendered frame with pest bounding boxes
- dataset summary artifacts
- evaluation summary and example failure cases


In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

from IPython.display import display
from PIL import Image, ImageDraw


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory.")


def choose_existing_path(*candidates: Path) -> Path:
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("None of the candidate paths exist: " + ", ".join(str(p) for p in candidates))


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def draw_bboxes(frame_path: Path, pests: list[dict]) -> Image.Image:
    image = Image.open(frame_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    color_map = {
        "mouse": (60, 140, 255),
        "rat": (255, 80, 60),
        "cockroach": (255, 190, 40),
    }
    for pest in pests:
        label = pest["label"]
        bbox = pest["bbox"]
        color = color_map.get(label, (255, 255, 255))
        box = [bbox["x_min"], bbox["y_min"], bbox["x_max"], bbox["y_max"]]
        draw.rectangle(box, outline=color, width=3)
        draw.text((bbox["x_min"], max(0, bbox["y_min"] - 14)), label, fill=color)
    return image


In [ ]:
repo_root = find_repo_root(Path.cwd())
config_path = repo_root / "configs" / "dcc_gpu.json"
config = load_json(config_path)

formal_batch_dir = repo_root / config["render"]["batch_output_dir"]
smoke_batch_dir = repo_root / "artifacts" / "batch_render_dcc_smoke"
batch_dir = choose_existing_path(formal_batch_dir, smoke_batch_dir)

dataset_summary_path = choose_existing_path(
    repo_root / "artifacts" / "dataset" / "dataset_summary.json",
)
evaluation_report_path = choose_existing_path(
    repo_root / "artifacts" / "reports" / "evaluation" / "detector_evaluation_report.json",
)
manifest_path = repo_root / config["inputs"]["kitchen_manifest"]

print(f"repo_root={repo_root}")
print(f"config_path={config_path}")
print(f"batch_dir={batch_dir}")
print(f"dataset_summary={dataset_summary_path}")
print(f"evaluation_report={evaluation_report_path}")


## 1. Input Kitchen Photo

The renderer starts from a real kitchen image listed in the manifest. We show the first rendered kitchen scene found under the batch output directory.

In [ ]:
scene_dirs = sorted([path for path in batch_dir.iterdir() if path.is_dir()])
if not scene_dirs:
    raise FileNotFoundError(f"No rendered scene directories found in {batch_dir}")

scene_dir = scene_dirs[0]
image_id = scene_dir.name

with manifest_path.open("r", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

manifest_row = next((row for row in rows if row["image_id"] == image_id), None)
if manifest_row is None:
    raise KeyError(f"Could not find {image_id} in manifest {manifest_path}")

input_image_path = repo_root / manifest_row["relative_path"]
print(f"image_id={image_id}")
print(f"input_image_path={input_image_path}")
display(Image.open(input_image_path))


## 2. Rendered Frame With Bounding Boxes

This cell reads the generated annotation JSON, finds the first frame containing pests, and overlays the stored bounding boxes.

In [ ]:
annotations_path = scene_dir / "annotations.json"
annotations = load_json(annotations_path)
frame_record = next((frame for frame in annotations if frame.get("pests")), None)
if frame_record is None:
    raise ValueError(f"No pest-containing frame found in {annotations_path}")

frame_filename = Path(frame_record["file"]).name
frame_path = scene_dir / "frames" / frame_filename
annotated_frame = draw_bboxes(frame_path, frame_record["pests"])

print(f"frame={frame_record['frame']}")
print(f"frame_path={frame_path}")
print("pests=")
for pest in frame_record["pests"]:
    bbox = pest["bbox"]
    print(f"  - {pest['label']}: [{bbox['x_min']}, {bbox['y_min']}, {bbox['width']}, {bbox['height']}]")

display(annotated_frame)


## 3. Dataset Packaging Summary

After rendering, the pipeline converts frame annotations into COCO and YOLO exports and prepares a real-image negative holdout.

In [ ]:
dataset_summary = load_json(dataset_summary_path)
print(json.dumps(dataset_summary, indent=2))


## 4. Evaluation Summary

The current detector baseline writes a validation/negative-holdout evaluation report after training.

In [ ]:
evaluation_report = load_json(evaluation_report_path)
print(f"model_name={evaluation_report['model_name']}")
print(f"thresholds={evaluation_report['thresholds']}")
print(f"iou_threshold={evaluation_report['iou_threshold']}")
print()

for split_name, split_payload in evaluation_report["splits"].items():
    print(f"[{split_name}]")
    print(f"  images={split_payload['images']}")
    for threshold, metrics in split_payload["threshold_table"].items():
        print(
            f"  threshold={threshold} "
            f"TDR={metrics['true_detection_rate']:.4f} "
            f"FPR={metrics['false_positive_rate']:.4f} "
            f"matched={metrics['matched_boxes']} / gt={metrics['ground_truth_boxes']}"
        )
    print()


## 5. Example Failure Cases

These examples come from the generated evaluation gallery and are useful in the submission appendix or FAQ.

In [ ]:
failure_examples = []
for split_name, split_payload in evaluation_report["splits"].items():
    examples = split_payload.get("failure_examples", [])
    if examples:
        failure_examples.append((split_name, examples[0]))

if not failure_examples:
    print("No failure examples were recorded in the evaluation report.")
else:
    for split_name, example in failure_examples:
        image_path = Path(example["visualization"])
        print(f"split={split_name} type={example['type']}")
        print(f"source={example['image']}")
        print(f"visualization={image_path}")
        display(Image.open(image_path))
